In [3]:
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace

data = ["Deep learning is fun", "AI is fun"]

# Build tokenizer (WorLevel) + whitespace split
tokenizer = Tokenizer(WordLevel(unk_token="<unk>"))
tokenizer.pre_tokenizer = Whitespace()

# Train vocab (lowercase to mimic basic_english)
trainer = WordLevelTrainer(
    vocab_size=7,
    special_tokens=["<unk>", "<pad>"]
)
tokenizer.train_from_iterator([s.lower() for s in data], trainer)

vocab = tokenizer.get_vocab()   # dict token -> id
vocab_ordered = dict(sorted(vocab.items(), key=lambda x: x[1])) # Sort by id (ascending)
print(vocab_ordered)

{'<unk>': 0, '<pad>': 1, 'fun': 2, 'is': 3, 'ai': 4, 'deep': 5, 'learning': 6}


In [4]:
sequence_length = 4

pad_id = tokenizer.token_to_id("<pad>")
tokenizer.enable_padding(pad_id=pad_id, pad_token="<pad>", length=sequence_length)
tokenizer.enable_truncation(max_length=sequence_length)

def vectorize_sentence(sentence: str):
    return tokenizer.encode(sentence.lower()).ids

vec1 = vectorize_sentence(data[0])
vec2 = vectorize_sentence(data[1])

print("vec1 = ", vec1)
print("vec2 =", vec2)

vec1 =  [5, 6, 3, 2]
vec2 = [4, 3, 2, 1]


In [7]:
import torch
import torch.nn as nn

vocab_size = len(vocab_ordered)
embedding = nn.Embedding(vocab_size, 2)

custom_weights = torch.tensor([
    [-0.1882, 0.5530],
    [1.7840, -0.8278],
    [1.0281, -1.9094],
    [-1.3083, -0.0987],
    [0.2293, 1.3255],
    [0.458, -0.6624],
    [0.5582, 0.0786],
]).float()
embedding.weight = nn.Parameter(custom_weights)

out1 = embedding(torch.tensor(vec1))
out2 = embedding(torch.tensor(vec2))

print("out1 =", out1)
print("out2 =", out2)

out1 = tensor([[ 0.4580, -0.6624],
        [ 0.5582,  0.0786],
        [-1.3083, -0.0987],
        [ 1.0281, -1.9094]], grad_fn=<EmbeddingBackward0>)
out2 = tensor([[ 0.2293,  1.3255],
        [-1.3083, -0.0987],
        [ 1.0281, -1.9094],
        [ 1.7840, -0.8278]], grad_fn=<EmbeddingBackward0>)


In [10]:
import numpy as np

def tanh(x):
    return (np.exp(x) - np.exp(-x)) / (np.exp(x) + np.exp(-x))

tanh(np.array([0.269, 0.585]))

array([0.26269409, 0.52629003])